In [1]:
from decouple import config
from sqlalchemy import create_engine
from sqlalchemy import text
import pandas as pd
import oracledb
import sys
from pyspark.sql import SparkSession

In [2]:
tabla='ctppe10'
col_essi='fec_pro'
col_tabla='properfec'
essi='essi_dat_pro001_etl'
col_dat='fec_pro'
dat='dat_progasis_essi'

In [3]:
# Oracle Conexion
DB_USER_ORACLE = config("USER4_BDI_POSTGRES")
DB_PASSWORD_ORACLE = config("PASS4_BDI_POSTGRES")
DB_NAME_ORACLE = "WNET"
DB_PORT_ORACLE = "1527"
DB_HOST_ORACLE = config("HOST4_BDI_POSTGRES")
cadena0 = f"jdbc:oracle:thin:@//{DB_HOST_ORACLE}:{DB_PORT_ORACLE}/{DB_NAME_ORACLE}"

# PostgreSQL Conexiones
DB_USER = config("USER2_BDI_POSTGRES")
DB_PASSWORD = config("PASS2_BDI_POSTGRES")
DB_PORT = "5432"
DB_HOST = config("HOST2_BDI_POSTGRES")

# Connection for essi_etl
DB_NAME_ESSI_ETL = "essi_etl"
url1 = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME_ESSI_ETL}"
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME_ESSI_ETL}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

# Connection for dw_essalud
DB_NAME_DW_ESSALUD = "dw_essalud"
url2 = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME_DW_ESSALUD}"
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME_DW_ESSALUD}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

# Connection for dl_essi
DB_NAME_DL_ESSI = "dl_essi"
url3 = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME_DL_ESSI}"
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME_DL_ESSI}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [4]:
spark = SparkSession.builder \
    .appName('SESION') \
    .config('spark.master', 'spark://10.0.27.37:7077') \
    .config("spark.executor.instances", "2") \
    .config("spark.executor.cores", "6") \
    .config("spark.executor.memory", "14g") \
    .config("spark.driver.memory", "14g") \
    .config("spark.sql.shuffle.partitions", "12") \
    .config("spark.default.parallelism", "12") \
    .getOrCreate()


23/08/29 17:01:33 WARN Utils: Your hostname, linux01 resolves to a loopback address: 127.0.0.1; using 10.0.27.37 instead (on interface eno1)
23/08/29 17:01:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/08/29 17:01:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
from datetime import datetime
from sqlalchemy import text

query = "SELECT fec_ini FROM etl_act where id_mod=1"

fecha = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()

fecha = fecha.collect()[0][0]

print(fecha)

now = datetime.now()

In [ ]:
fecha='2023-08-28'

In [ ]:

query = f"SELECT * FROM {tabla} WHERE {col_tabla} >= TO_DATE('{fecha}', 'YYYY-MM-DD')"

lake = spark.read \
    .format("jdbc") \
    .option("driver", "oracle.jdbc.OracleDriver" ) \
    .option("url", cadena0) \
    .option("query", query) \
    .option("user", DB_USER_ORACLE) \
    .option("password", DB_PASSWORD_ORACLE) \
    .load()



In [ ]:
# Convirtiendo los nombres de las columnas a minúsculas
for col in lake.columns:
    lake = lake.withColumnRenamed(col, col.lower())

In [ ]:
lake.cache()

In [ ]:
lake.count()

In [ ]:
lake.show(5)

In [ ]:
borrando=f"DELETE FROM {tabla} WHERE {col_tabla} >='{fecha}'"
borrado = engine3.execute(borrando)

In [ ]:
lake.write \
    .format("jdbc") \
    .option("url", url3) \
    .option("dbtable", tabla) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [ ]:
query = f"SELECT * FROM {dat} LIMIT 10"
base2 = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()

In [ ]:
query = f"SELECT id_oricas,ori_cod FROM dim_oricas"
ori = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
ori = ori.withColumnRenamed('ori_cod', 'cod_ori')
ori = ori.withColumnRenamed('id_oricas', 'id_origen')
base1 = base1.withColumnRenamed('oricenasicod', 'cod_ori')
base1 = base1.merge(ori, how="inner", on="cod_ori")
base1 = base1.drop('cod_ori')

In [ ]:
query = f"SELECT id_cas, id_red, cod_cas FROM dim_cas where id_red is not null"
cas = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
cas = cas.dropna()
base1 = base1.withColumnRenamed('cenasicod', 'cod_cas')
base1 = base1.merge(cas, how="inner", on="cod_cas")
base1 = base1.drop('cod_cas')

In [ ]:
query = f"SELECT id_serv,cod_ser FROM dim_servicios"
serv = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
serv = serv.withColumnRenamed('id_serv', 'id_servic')
base1 = base1.withColumnRenamed('servhoscod', 'cod_ser')
base1 = base1.join(serv, on="cod_ser", how="inner")
base1 = base1.drop('cod_ser')

In [ ]:
query = f"SELECT id_area,cod_are FROM dim_areas"
are = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
base1 = base1.withColumnRenamed('arehoscod', 'cod_are')
base1 = base1.join(are, on="cod_are", how="inner")
base1=base1.drop('cod_are')

In [ ]:
query = f"SELECT id_activi,cod_act FROM dim_activi"
activi = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
activi = activi.withColumnRenamed('id_activi', 'id_acti')
base1 = base1.withColumnRenamed('actcod', 'cod_act')
base1 = base1.join(activi, how='inner', on='cod_act')

In [ ]:
query = f"SELECT id_subacti,cod_sub,cod_act FROM dim_subacti"
subacti = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
subacti = subacti.withColumn("KEY", concat(subacti["cod_sub"], subacti["cod_act"]))
subacti = subacti.drop("cod_sub",'cod_act')
base1 = base1.withColumnRenamed('actespcod', 'cod_sub')
base1 = base1.withColumn("KEY", concat(col("cod_sub").cast("string"), col("cod_act").cast("string")))
base1 = base1.withColumn("KEY", regexp_replace(base1["KEY"], " ", ""))
subacti = subacti.withColumn("KEY", regexp_replace(subacti["KEY"], " ", ""))
base1 = base1.join(subacti, how='inner', on='KEY')
base1 = base1.drop('KEY')
base1 = base1.drop('cod_act')
base1 = base1.drop('cod_sub')


In [ ]:
query = f"SELECT id_tipdoc,cod_tdo FROM dim_tipdoc"
tipdoc = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
tipdoc = tipdoc.withColumnRenamed('cod_tdo', 'tdo_pro')
base1 = base1.withColumnRenamed('tipdocidenpercod', 'tdo_pro')
base1 = base1.join(tipdoc, how='inner', on='tdo_pro')
base1 = base1.drop('tdo_pro')

In [ ]:
query = f"SELECT id_person,num_doc FROM dim_personal"
personal = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
personal = personal.withColumnRenamed('id_person', 'id_profesional')
personal = personal.withColumnRenamed('num_doc', 'dni_pro')
personal = personal.orderBy(desc("id_profesional"))
personal = personal.withColumn("dni_pro", regexp_replace("dni_pro", " ", ""))
personal = personal.dropDuplicates(["dni_pro"])
base1 = base1.withColumnRenamed('perasisdocidennum', 'dni_pro')
base1 = base1.withColumn("dni_pro", regexp_replace("dni_pro", " ", ""))
base1 = base1.join(personal, how='inner', on='dni_pro')
base1 = base1.drop('dni_pro')

In [ ]:
query = f"SELECT id_tipdoc,cod_tdo FROM dim_tipdoc"
tipdoc = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
tipdoc = tipdoc.withColumnRenamed('cod_tdo', 'tdo_pro')
base1 = base1.withColumnRenamed('tipdocidenpercod', 'tdo_pro')
base1 = base1.join(tipdoc, how='inner', on='tdo_pro')
base1 = base1.drop('tdo_pro')

In [ ]:
query = f"SELECT id_tipcuphor,cod_tch FROM dim_tipcuphor"
tipcuphor = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
tipcuphor = tipcuphor.withColumnRenamed('cod_tch', 'cod_pro')
tipcuphor = tipcuphor.withColumnRenamed('id_tipcuphor', 'id_tippro')
base1 = base1.withColumnRenamed('propertipoprogperscod', 'cod_pro')
base1 = base1.join(tipcuphor, how='inner', on='cod_pro')
base1 = base1.drop('cod_pro')

In [ ]:
query = f"SELECT id_motsus,cod_mot FROM dim_motsuspro"
motsuspro = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
motsuspro = motsuspro.withColumnRenamed('cod_mot', 'cod_sus')
base1 = base1.withColumnRenamed('motsusprogcod', 'cod_sus')
base1 = base1.join(motsuspro, how='left', on='cod_sus')
base1 = base1.drop('cod_sus')

In [ ]:
base1 = base1.withColumnRenamed('hor_ter', 'hor_fin')

In [ ]:
# base1 = base1.withColumn('hor_inix', substring(base1['hor_ini'].cast('string'), 11, 30))
# base1 = base1.withColumn("hor_finx", substring(base1["hor_fin"].cast("string"), 11, 30))
# base1 = base1.withColumn("hor_inix", substring(base1["hor_inix"], 1, 6))
# base1 = base1.withColumn("hor_finx", substring(base1["hor_finx"], 1, 6))
# base1 = base1.withColumn("hor_finx", regexp_replace("hor_finx", " ", ""))
# base1 = base1.withColumn("hor_inix", regexp_replace("hor_inix", " ", ""))
# base1 = base1.withColumn("turno", concat(base1["hor_inix"].cast("string"), lit("-"), base1["hor_finx"].cast("string")))

In [ ]:
# query = f"SELECT id_turno,horas,minutos,destur,horario FROM dim_turnos"
# turnos = spark.read \
#     .format("jdbc") \
#     .option("url", url2)\
#     .option("query", query) \
#     .option("user", DB_USER) \
#     .option("password", DB_PASSWORD) \
#     .option("driver", "org.postgresql.Driver") \
#     .load()
# turnos = turnos.withColumnRenamed('destur', 'turno')
# turnos = turnos.withColumn("turno", regexp_replace(col("turno"), " ", ""))
# merge = base1.join(turnos, how='left', on='turno')
# base1 = base1.join(turnos, how='inner', on='turno')

In [ ]:
#log_falla('id_turno', 'turno', True)
#log_control('dim_turnos')
# base1=base1.drop('turno')
# base1.count()

In [ ]:
# query = f"SELECT id_horario, cod_hor FROM dim_horario"
# horario = spark.read \
#     .format("jdbc") \
#     .option("url", url2)\
#     .option("query", query) \
#     .option("user", DB_USER) \
#     .option("password", DB_PASSWORD) \
#     .option("driver", "org.postgresql.Driver") \
#     .load()
# horario = horario.withColumnRenamed('cod_hor', 'horario')
# merge = base1.join(horario, how='left', on='horario')
# base1 = base1.join(horario, how='inner', on='horario')

In [ ]:
# #log_falla('id_horario', 'horario', True)
# #log_control('dim_horario')
# base1=base1.drop('horario')

In [ ]:
query = f"SELECT id_estsolcit,cod_esc FROM dim_cexestsolcit"
estsolcit = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
estsolcit = estsolcit.withColumnRenamed('id_estsolcit', 'id_tci')
estsolcit = estsolcit.withColumnRenamed('cod_esc', 'cod_tci')
merge = base1.join(estsolcit, how='left', on='cod_tci')
base1 = base1.join(estsolcit, how='inner', on='cod_tci')

In [ ]:
#log_falla('id_tci', 'cod_tci', True)
#log_control('dim_cexestsolcit')
base1=base1.drop('cod_tci')

In [ ]:
query = f"SELECT id_tipocit,cod_tci FROM dim_tipcita"
tipcita = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
tipcita = tipcita.withColumnRenamed('id_tipocit', 'id_cci')
tipcita = tipcita.withColumnRenamed('cod_tci', 'cod_cci')
merge = base1.join(tipcita, how='left', on='cod_cci')
base1 = base1.join(tipcita, how='inner', on='cod_cci')

In [ ]:
#log_falla('id_cci', 'cod_cci', True)
#log_control('dim_tipcita')
base1=base1.drop('cod_cci')

In [ ]:
query = f"SELECT id_tipemi,cod_emi FROM dim_tipemi"
tipemi = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
tipemi = tipemi.withColumnRenamed('id_tipemi', 'id_otorga')
tipemi = tipemi.withColumnRenamed('cod_emi', 'cod_oto')
merge = base1.join(tipemi, how='left', on='cod_oto')
base1 = base1.join(tipemi, how='inner', on='cod_oto')

In [ ]:
#log_falla('id_otorga', 'cod_oto', True)
#log_control('dim_tipemi')
base1=base1.drop('cod_oto')

In [ ]:
base1 = base1.withColumnRenamed('cit_rep', 'id_citrep')

In [ ]:
query = f"SELECT id_moteli,cod_eli FROM dim_cexmotelicit"
mec = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
mec = mec.withColumnRenamed('id_moteli', 'id_mec')
mec = mec.withColumnRenamed('cod_eli', 'cod_mec')
merge = base1.join(mec, how='left', on='cod_mec')
base1 = base1.join(mec, how='left', on='cod_mec')

In [ ]:
#log_falla('id_mec', 'cod_mec', True)
#log_control('dim_cexmotelicit')
base1=base1.drop('cod_mec')

In [ ]:
query = f"SELECT id_estcit,cod_eci FROM dim_estcit"
eci = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
eci = eci.withColumnRenamed('id_estcit', 'id_eci')
merge = base1.join(eci, how='left', on='cod_eci')
base1 = base1.join(eci, how='inner', on='cod_eci')

In [ ]:
#log_falla('id_eci', 'cod_eci', True)
#log_control('dim_estcit')
base1=base1.drop('cod_eci')

In [ ]:
query = f"SELECT id_esteco,cod_eco FROM dim_cexcitoto"
enco = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
enco = enco.withColumnRenamed('id_esteco', 'id_enco')
enco = enco.withColumnRenamed('cod_eco', 'cod_enco')
merge = base1.join(enco, how='left', on='cod_enco')
base1 = base1.join(enco, how='inner', on='cod_enco')

In [ ]:
#log_falla('id_enco', 'cod_enco', True)
base1=base1.drop('cod_enco')
#dim.append('dim_cexcitoto')
#control_d.append(base1.count())

In [ ]:
base1 = base1.withColumnRenamed('pac_sec', 'id_pacsec')

In [ ]:
# base1 = base1.withColumn('horfin', trim(col('hor_finx').cast(StringType())))
# base1 = base1.withColumn('horfin', to_timestamp('horfin', 'HH:mm'))
# base1 = base1.withColumn('horini', trim(col('hor_inix').cast(StringType())))
# base1 = base1.withColumn('horini', to_timestamp('horini', 'HH:mm'))
# base1 = base1.withColumn('diferencia_horas', (col('horfin').cast('long') - col('horini').cast('long')).cast(StringType()))
# base1 = base1.withColumn('horas', base1['diferencia_horas'].substr(-9, 3))

In [ ]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

In [ ]:
# query = f"DELETE FROM {dat} WHERE {col_dat} >='{fecha}'"
# borrado = spark.read \
#     .format("jdbc") \
#     .option("url", url2)\
#     .option("query", query) \
#     .option("user", DB_USER) \
#     .option("password", DB_PASSWORD) \
#     .option("driver", "org.postgresql.Driver") \
#     .load()

In [ ]:
comunes = set(base1.columns).intersection(set(base2.columns))
base = base1.select(*comunes)

In [ ]:
url2 = f"jdbc:postgresql://10.0.27.41:{DB_PORT}/temporal"
base.write \
    .format("jdbc") \
    .option("url", url2) \
    .option("dbtable", f"tmp_dat_spark") \
    .option("user", 'ugading003jc') \
    .option("password", 'U64dJC2023') \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()
now_fin = datetime.now()